In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00


In [2]:
import torch
import json
import re
from pathlib import Path

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

print("✅ MODEL LOADED")


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
TIMESTAMP_REGEX = re.compile(r"^\[\d\d:\d\d:\d\d\.\d+\]\s*")

def clean_transcription_lines(lines, additional_info = []):
    """
    Input: list[str] with timestamps, str extra text
    Output: one large cleaned string
    """

    cleaned = []

    for line in lines:
        # remove timestamp
        text = TIMESTAMP_REGEX.sub("", line).strip()

        if text:
            cleaned.append(text)

    cleaned.extend(additional_info)

    return "\n".join(cleaned)


In [ ]:
def chunk_text(text, max_chars=900):

    chunks = []
    current = ""

    for line in text.split("\n"):
        if len(current) + len(line) > max_chars:
            chunks.append(current)
            current = line + "\n"
        else:
            current += line + "\n"

    if current.strip():
        chunks.append(current)

    return chunks


In [ ]:
def extract_ingredients_chunk(text):

    prompt = f"""
You are an information extraction system.

Extract ingredients from the cooking transcript below.

Rules:
- If none, return {{ "ingredients": [] }}
- Return ONLY valid JSON
- lowercase
- remove quantities
- no commentary

Text:
{text}

JSON:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.1,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # grab LAST json block only
    matches = re.findall(r"\{[\s\S]*?\}", decoded)

    if not matches:
        return []

    try:
        parsed = json.loads(matches[-1])
        return parsed.get("ingredients", [])
    except:
        return []


In [ ]:
def extract_from_large_text(text):

    chunks = chunk_text(text)

    all_items = []

    for chunk in chunks:
        items = extract_ingredients_chunk(chunk)
        all_items.extend(items)

    # raw merged list
    return list(set(all_items))


In [ ]:
def cleanup_ingredient_list(raw_list):

    if not raw_list:
        return []

    joined = ", ".join(raw_list)
    example = "{ \"ingredients\": [\"item1\", \"item2\"] }"
    empty_example = "{ \"ingredients\": [] }"
    prompt = f"""
You are cleaning a noisy ingredient list extracted from speech.

Input list:
[{joined}]

Your task:

- Remove anything that is NOT a food ingredient
- Remove restaurants, places, people, brands
- Remove nonsense or gibberish
- Deduplicate items
- Correct spelling mistakes
- Normalize to singular nouns where possible
- Lowercase everything
- If uncertain, REMOVE rather than guess

Return ONLY valid JSON with this structure:
ingredients must be a list of strings.
Format to return: {example}
If there are no ingredients, return {empty_example}
Do NOT include commentary or examples, only real data in the above shown format
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=350,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # extract LAST JSON-looking object only
    matches = re.findall(r"\{[\s\S]*?\}", decoded)

    if not matches:
        print("⚠️ Cleanup pass returned no JSON")
        return []

    json_text = matches[-1]

    try:
        parsed = json.loads(json_text)

        ingredients = parsed.get("ingredients", [])

        # ---- sanity check: reject placeholder junk ----
        banned = {"item1", "item2", "example", "ingredient"}

        ingredients = [
            x for x in ingredients
            if isinstance(x, str)
            and x.strip().lower() not in banned
            and len(x.strip()) > 2
        ]

        return ingredients

    except Exception as e:
        print("⚠️ Cleanup JSON parse failed:", e)
        return []


In [ ]:
def process_file(file_path):

    file_path = Path(file_path)

    # -----------------
    # Load JSON
    # -----------------

    try:
        with open(file_path, "r") as f:
            data = json.load(f)
    except Exception:
        return False

    if "transcription_english" not in data:
        return False

    # -----------------
    # Load Title and Description
    # -----------------

    additional_info = []

    if "title" in data["metadata"]:
        additional_info.append(data["metadata"]["title"])

    if "description" in data["metadata"]:
        additional_info.append(data["metadata"]["description"])

    # -----------------
    # Clean transcript
    # -----------------

    transcript_text = clean_transcription_lines(
        data["transcription_english"],
        additional_info
    )

    # -----------------
    # First pass extraction
    # -----------------

    raw_items = extract_from_large_text(transcript_text)

    # -----------------
    # Second pass cleanup
    # -----------------

    cleaned_items = cleanup_ingredient_list(raw_items)

    # -----------------
    # Write back into metadata
    # -----------------

    if "metadata" not in data or not isinstance(data["metadata"], dict):
        data["metadata"] = {}

    data["metadata"]["ingredients_detected"] = cleaned_items

    # -----------------
    # Save file
    # -----------------

    with open(file_path, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    return True


In [ ]:
QUEUE_DIR = Path("/content/drive/MyDrive/data")

TO_PROCESS_FILE = QUEUE_DIR / "to_process.txt"
PROCESSED_FILE = QUEUE_DIR / "processed.txt"


In [ ]:
def initialize_processing_queue(data_root="/content/drive/MyDrive/data"):

    data_root = Path(data_root)

    print("🔍 Scanning for JSON files...")

    all_json_files = list(data_root.rglob("*.json"))

    print(f"Found {len(all_json_files)} total json files")

    valid = []

    for p in all_json_files:
        try:
            with open(p, "r") as f:
                data = json.load(f)

            if "transcription_english" in data:
                valid.append(str(p.resolve()))

        except Exception:
            continue

    print(f"Filtered to {len(valid)} files with transcription_english")

    # write to to_process.txt
    TO_PROCESS_FILE.write_text("\n".join(valid))

    # create processed.txt empty
    PROCESSED_FILE.write_text("")

    print("✅ Queue initialized")


In [ ]:
def load_queue():

    to_process = []
    processed = []

    if TO_PROCESS_FILE.exists():
        to_process = [
            x.strip()
            for x in TO_PROCESS_FILE.read_text().splitlines()
            if x.strip()
        ]

    if PROCESSED_FILE.exists():
        processed = [
            x.strip()
            for x in PROCESSED_FILE.read_text().splitlines()
            if x.strip()
        ]

    return to_process, processed


In [ ]:
def save_queue(to_process, processed):

    TO_PROCESS_FILE.write_text("\n".join(to_process))
    PROCESSED_FILE.write_text("\n".join(processed))


In [ ]:
from tqdm import tqdm

def run_batch_from_queue():

    to_process, processed = load_queue()

    print(f"📌 Remaining: {len(to_process)} | Done: {len(processed)}")

    if not to_process:
        print("🎉 Nothing left to process!")
        return

    for path in tqdm(list(to_process)):

        success = process_file(path)

        if success:
            to_process.remove(path)
            processed.append(path)

            # persist every step
            save_queue(to_process, processed)


In [ ]:
# first time only
# initialize_processing_queue("/content/drive/MyDrive/data")


In [ ]:
run_batch_from_queue()
